# Geometric EEG SSL — Data Download Notebook (CPU only)

**Purpose:** Cache all three datasets to Google Drive once. No GPU needed —
**do not waste GPU credit running this**. Use a standard CPU runtime.

After this notebook completes:
- PhysioNet MI is enough to start **`colab_pretrain.ipynb`** (all 5 variants).
- BCIC-2B and Sleep-EDFx are only required to run **`colab_experiment.ipynb`**.

Sections **4a / 4b / 4c** are independent — you can run them in any order or
re-run individually. All three are resume-aware: already-cached files are skipped.

---

## 1. Install dependencies

In [ ]:
%%capture
!pip install mne moabb scikit-learn pyyaml scipy

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data cached here — avoids re-downloading across sessions
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.makedirs(MNE_DATA_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Checkpoints saved here
CKPT_ROOT = f'{DRIVE_ROOT}/runs'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Drive mounted. Checkpoints → {CKPT_ROOT}')

## 3. Clone repo (optional, for the loader code)

In [ ]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR)

## 4a. Download PhysioNet MI data

In [ ]:
import os, mne
mne.set_log_level('WARNING')

EXCLUDED = {88, 92, 100, 104}
ALL_SUBJECTS_MI = [s for s in range(1, 110) if s not in EXCLUDED]  # 105 subjects
MI_RUNS = [4, 6, 8, 10, 12, 14]

EEGBCI_ROOT = os.path.join(MNE_DATA_DIR, 'MNE-eegbci-data', 'files', 'eegmmidb', '1.0.0')

def subject_fully_cached(subj, runs):
    subj_dir = os.path.join(EEGBCI_ROOT, f'S{subj:03d}')
    if not os.path.isdir(subj_dir):
        return False
    return all(
        os.path.exists(os.path.join(subj_dir, f'S{subj:03d}R{run:02d}.edf')) and
        os.path.getsize(os.path.join(subj_dir, f'S{subj:03d}R{run:02d}.edf')) > 0
        for run in runs
    )

cached = [s for s in ALL_SUBJECTS_MI if subject_fully_cached(s, MI_RUNS)]
todo = [s for s in ALL_SUBJECTS_MI if s not in cached]
print(f'PhysioNet MI: cached {len(cached)}/{len(ALL_SUBJECTS_MI)} subjects. Downloading {len(todo)}.')

for i, subj in enumerate(todo):
    try:
        mne.datasets.eegbci.load_data(subj, MI_RUNS, path=MNE_DATA_DIR, verbose=False)
    except Exception as e:
        print(f'  subject {subj:3d}: download failed ({e})')
        continue
    if (i + 1) % 10 == 0 or (i + 1) == len(todo):
        print(f'  downloaded {i+1}/{len(todo)} (subject {subj:3d})')

still_missing = [s for s in ALL_SUBJECTS_MI if not subject_fully_cached(s, MI_RUNS)]
if still_missing:
    print(f'WARNING: {len(still_missing)} subjects still incomplete: {still_missing}')
else:
    print(f'All {len(ALL_SUBJECTS_MI)} PhysioNet MI subjects ready.')

## 4b. Download BCIC-2B data

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR

from moabb.datasets import BNCI2014_004
import moabb
moabb.set_log_level('WARNING')

ds = BNCI2014_004()
print('Downloading BCIC-2B (BNCI2014_004)...')
try:
    ds.download(subject_list=list(range(1, 10)))
    print('BCIC-2B download complete.')
except Exception as e:
    # MOABB sometimes raises on partial cache; data may still be usable
    print(f'MOABB download reported: {e}')
    print('Attempting to load subject 1 to verify cache...')
    try:
        _ = ds.get_data(subjects=[1])
        print('Subject 1 loaded OK — cache is usable.')
    except Exception as e2:
        print(f'WARNING: could not load subject 1: {e2}')

## 4c. Download Sleep-EDFx data

In [ ]:
import os, mne
mne.set_log_level('WARNING')
os.environ['MNE_DATA'] = MNE_DATA_DIR

# MNE stores Sleep-EDFx under {MNE_DATA}/physionet-sleep-data/
_UNAVAILABLE = {39, 68, 69, 78, 79}
ALL_SUBJECTS_SLEEP = [s for s in range(0, 83) if s not in _UNAVAILABLE]
print(f'Sleep-EDFx: {len(ALL_SUBJECTS_SLEEP)} subjects to cache.')

n_done = 0
n_failed = 0
for subj in ALL_SUBJECTS_SLEEP:
    try:
        mne.datasets.sleep_physionet.age.fetch_data(
            subjects=[subj], recording=[1, 2], path=MNE_DATA_DIR, verbose=False
        )
        n_done += 1
    except Exception as e:
        n_failed += 1
        # Some subjects have only 1 recording — try night 1 only
        try:
            mne.datasets.sleep_physionet.age.fetch_data(
                subjects=[subj], recording=[1], path=MNE_DATA_DIR, verbose=False
            )
            n_done += 1
            n_failed -= 1
        except Exception:
            pass

print(f'Sleep-EDFx: {n_done} subjects cached, {n_failed} failed/unavailable.')

## Done

Close this runtime to free CPU resources. Open `colab_pretrain.ipynb` with a
**GPU runtime** to start training (PhysioNet MI alone is sufficient; the
other two datasets are only needed by the experiment notebook).